In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import yaml

In [2]:
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [3]:
final_demo = pl.read_csv(config["input_data"]["file1"])
final_demo.head()

client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
i64,f64,f64,f64,str,f64,f64,f64,f64
836976,6.0,73.0,60.5,"""U""",2.0,45105.3,6.0,9.0
2304905,7.0,94.0,58.0,"""U""",2.0,110860.3,6.0,9.0
1439522,5.0,64.0,32.0,"""U""",2.0,52467.79,6.0,9.0
1562045,16.0,198.0,49.0,"""M""",2.0,67454.65,3.0,6.0
5126305,12.0,145.0,33.0,"""F""",2.0,103671.75,0.0,3.0


In [4]:
experiment_data = pl.read_csv(config["input_data"]["file2"])
experiment_data.head()

client_id,Variation
i64,str
9988021,"""Test"""
8320017,"""Test"""
4033851,"""Control"""
1982004,"""Test"""
9294070,"""Control"""


In [5]:
webdata1 = pl.read_csv(config["input_data"]["file3"])
webdata1.head()

client_id,visitor_id,visit_id,process_step,date_time
i64,str,str,str,str
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:27:07"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:26:51"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:19:22"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:19:13"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:18:04"""


In [6]:
webdata2 = pl.read_csv(config["input_data"]["file2"])
webdata2.head()

client_id,Variation
i64,str
9988021,"""Test"""
8320017,"""Test"""
4033851,"""Control"""
1982004,"""Test"""
9294070,"""Control"""


In [7]:
webdata2["Variation"].value_counts()

Variation,count
str,u32
"""Test""",26968
"""Control""",23532
"""NA""",20109


In [8]:
webdata1["process_step"].value_counts().sort("count", descending=True)

process_step,count
str,u32
"""start""",108910
"""step_1""",73432
"""step_2""",61768
"""step_3""",53628
"""confirm""",45403


MERGED DATA

In [9]:
merged = pl.read_csv(config["output_data"]["file5"])
pl.DataFrame(merged).head()

,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation
i64,i64,str,str,str,str,f64,f64,f64,str,f64,f64,f64,f64,str
0,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:27:07""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
1,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:26:51""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
2,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:19:22""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
3,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:19:13""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
4,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:18:04""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""


Defining SEQ steps

In [10]:
df = pl.read_csv(config["output_data"]["file5"])

STEPS = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
STEP_ORDER = {step: i for step, i in zip(STEPS, range(len(STEPS)))}


In [11]:
print([x for x in dir() if not x.startswith("_")])

['In', 'Out', 'STEPS', 'STEP_ORDER', 'config', 'df', 'exit', 'experiment_data', 'file', 'final_demo', 'get_ipython', 'merged', 'open', 'pd', 'pl', 'plt', 'quit', 'sns', 'webdata1', 'webdata2', 'yaml']


In [12]:
client_visit_counts = (
    df
    .group_by(["Variation", "client_id"])
    .agg(
        pl.col("visit_id").n_unique().alias("n_visits")
    )
    .group_by(["Variation", "n_visits"])
    .agg(pl.len().alias("n_clients"))
    .sort(["Variation", "n_visits"])
)

print(client_visit_counts)

shape: (23, 3)
┌───────────┬──────────┬───────────┐
│ Variation ┆ n_visits ┆ n_clients │
│ ---       ┆ ---      ┆ ---       │
│ str       ┆ u32      ┆ u32       │
╞═══════════╪══════════╪═══════════╡
│ Control   ┆ 1        ┆ 14197     │
│ Control   ┆ 2        ┆ 2905      │
│ Control   ┆ 3        ┆ 614       │
│ Control   ┆ 4        ┆ 164       │
│ Control   ┆ 5        ┆ 56        │
│ …         ┆ …        ┆ …         │
│ Test      ┆ 6        ┆ 31        │
│ Test      ┆ 7        ┆ 8         │
│ Test      ┆ 8        ┆ 11        │
│ Test      ┆ 9        ┆ 2         │
│ Test      ┆ 10       ┆ 1         │
└───────────┴──────────┴───────────┘


Day 2, organizing thoughts

In [13]:
import polars as pl

STEP_ORDER = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
REQUIRED_STEPS = set(STEP_ORDER.keys())
STEPS = ["start", "step_1", "step_2", "step_3", "confirm"]

df = (
    pl.read_csv(config["output_data"]["file5"])
    .with_columns(pl.col("process_step").replace(STEP_ORDER).alias("step_rank"))
    .sort(["visit_id", "date_time"])
    .with_columns([
        pl.col("step_rank").shift(1).over("visit_id").alias("prev_step_rank"),
        pl.col("process_step").shift(1).over("visit_id").alias("prev_step_name"),
    ])
    .with_columns([
        (pl.col("step_rank") == pl.col("prev_step_rank")).alias("is_repetition"),
        (pl.col("step_rank") < pl.col("prev_step_rank")).alias("is_regression"),
    ])
    .with_columns(
        (pl.col("is_repetition") | pl.col("is_regression")).alias("is_error")
    )
)

def classify_visit(steps, step_ranks, had_error):
    if not REQUIRED_STEPS.issubset(set(steps)):
        return "incomplete"
    first_ranks = [int(step_ranks[next(i for i, s in enumerate(steps) if s == step)]) for step in STEPS]
    if not all(first_ranks[i] < first_ranks[i+1] for i in range(len(first_ranks) - 1)):
        return "incomplete"
    return "smooth" if not had_error else "lumpy"

visit_agg = (
    df.group_by("visit_id").agg([
        pl.col("process_step").alias("steps_list"),
        pl.col("step_rank").alias("step_ranks_list"),
        pl.col("date_time").min().alias("visit_start"),
        pl.col("date_time").max().alias("visit_end"),
        pl.col("is_error").any().alias("had_error"),
        pl.col("is_repetition").sum().alias("n_repetitions"),
        pl.col("is_regression").sum().alias("n_regressions"),
        pl.col("Variation").first().alias("variation"),
        pl.col("client_id").first().alias("client_id"),
        pl.col("visitor_id").first().alias("visitor_id"),
    ])
    .with_columns(
        pl.struct(["steps_list", "step_ranks_list", "had_error"])
          .map_elements(
              lambda r: classify_visit(r["steps_list"], r["step_ranks_list"], r["had_error"]),
              return_dtype=pl.String
          )
          .alias("completion_type")
    )
)

print(visit_agg.group_by(["variation", "completion_type"]).agg(pl.len().alias("n_visits")).sort(["variation", "completion_type"]))

shape: (6, 3)
┌───────────┬─────────────────┬──────────┐
│ variation ┆ completion_type ┆ n_visits │
│ ---       ┆ ---             ┆ ---      │
│ str       ┆ str             ┆ u32      │
╞═══════════╪═════════════════╪══════════╡
│ Control   ┆ incomplete      ┆ 12373    │
│ Control   ┆ lumpy           ┆ 3859     │
│ Control   ┆ smooth          ┆ 7037     │
│ Test      ┆ incomplete      ┆ 14444    │
│ Test      ┆ lumpy           ┆ 5954     │
│ Test      ┆ smooth          ┆ 8331     │
└───────────┴─────────────────┴──────────┘


For lumpy the rule was:

- All 5 steps must appear
- When you take only the first occurrence of each step, they must be in the correct order (start → 1 → 2 → 3 → confirm)
- Repetitions and regressions are allowed in between

So start-1-2-1-2-3-confirm IS OK but start-2-1-3-confirm IS NOT (first occurrence of 2 comes before 1).

Finding out completion in one visit (smooth + lumpy, by variation)

In [14]:
# A "one visit" completion means visit_id has all 5 steps — already captured in smooth/lumpy
one_visit = (
    visit_agg
    .filter(pl.col("completion_type").is_in(["smooth", "lumpy"]))
    .group_by(["variation", "completion_type"])
    .agg(pl.len().alias("n_visits"))
    .sort(["variation", "completion_type"])
)
print(one_visit)

shape: (4, 3)
┌───────────┬─────────────────┬──────────┐
│ variation ┆ completion_type ┆ n_visits │
│ ---       ┆ ---             ┆ ---      │
│ str       ┆ str             ┆ u32      │
╞═══════════╪═════════════════╪══════════╡
│ Control   ┆ lumpy           ┆ 3859     │
│ Control   ┆ smooth          ┆ 7037     │
│ Test      ┆ lumpy           ┆ 5954     │
│ Test      ┆ smooth          ┆ 8331     │
└───────────┴─────────────────┴──────────┘


Completed on first visit (first visit per client)

In [15]:
# For each client, find their chronologically first visit
# then check if that first visit was a completion

first_visits = (
    visit_agg
    .sort("visit_start")
    .group_by("client_id")
    .agg([
        pl.col("visit_id").first().alias("first_visit_id"),
        pl.col("completion_type").first().alias("first_visit_completion"),
        pl.col("variation").first().alias("variation"),
    ])
)

first_visit_completions = (
    first_visits
    .filter(pl.col("first_visit_completion").is_in(["smooth", "lumpy"]))
    .group_by(["variation", "first_visit_completion"])
    .agg(pl.len().alias("n_clients"))
    .sort(["variation", "first_visit_completion"])
)
print(first_visit_completions)

shape: (4, 3)
┌───────────┬────────────────────────┬───────────┐
│ variation ┆ first_visit_completion ┆ n_clients │
│ ---       ┆ ---                    ┆ ---       │
│ str       ┆ str                    ┆ u32       │
╞═══════════╪════════════════════════╪═══════════╡
│ Control   ┆ lumpy                  ┆ 3279      │
│ Control   ┆ smooth                 ┆ 6306      │
│ Test      ┆ lumpy                  ┆ 5094      │
│ Test      ┆ smooth                 ┆ 7357      │
└───────────┴────────────────────────┴───────────┘


Errors per step transition

In [16]:
# Errors per step transition — ranked buggiest to least buggy per group
RANK_TO_STEP = {"0": "start", "1": "step_1", "2": "step_2", "3": "step_3", "4": "confirm"}

buggiest_ranked = (
    df
    .filter(pl.col("is_error"))
    .with_columns(
        pl.col("prev_step_rank").replace(RANK_TO_STEP).alias("prev_step_rank")
    )
    .group_by(["Variation", "prev_step_rank", "process_step"])
    .agg([
        pl.col("is_repetition").cast(pl.Int32).sum().alias("n_repetitions"),
        pl.col("is_regression").cast(pl.Int32).sum().alias("n_regressions"),
        pl.len().alias("n_total_errors"),
    ])
    .with_columns(
        (pl.col("prev_step_rank") + pl.lit(" -> ") + pl.col("process_step"))
        .alias("transition")
    )
    .drop(["prev_step_rank", "process_step"])
    .sort(["Variation", "n_total_errors"], descending=[False, True])
    .with_columns(
        pl.int_range(pl.len()).over("Variation").alias("rank") + 1
    )
    .select(["Variation", "rank", "transition", "n_repetitions", "n_regressions", "n_total_errors"])
    .sort(["Variation", "rank"])
)
print(buggiest_ranked)

# Clear conclusion per group
for variation in ["Control", "Test"]:
    top = buggiest_ranked.filter(pl.col("Variation") == variation).head(1)
    print(f"\n[{variation}] Buggiest transition: '{top['transition'][0]}' — {top['n_total_errors'][0]} total errors "
          f"({top['n_repetitions'][0]} repetitions, {top['n_regressions'][0]} regressions)")

shape: (29, 6)
┌───────────┬──────┬───────────────────┬───────────────┬───────────────┬────────────────┐
│ Variation ┆ rank ┆ transition        ┆ n_repetitions ┆ n_regressions ┆ n_total_errors │
│ ---       ┆ ---  ┆ ---               ┆ ---           ┆ ---           ┆ ---            │
│ str       ┆ i64  ┆ str               ┆ i32           ┆ i32           ┆ u32            │
╞═══════════╪══════╪═══════════════════╪═══════════════╪═══════════════╪════════════════╡
│ Control   ┆ 1    ┆ start -> start    ┆ 6118          ┆ 0             ┆ 6118           │
│ Control   ┆ 2    ┆ step_3 -> step_2  ┆ 0             ┆ 1946          ┆ 1946           │
│ Control   ┆ 3    ┆ step_1 -> start   ┆ 0             ┆ 1152          ┆ 1152           │
│ Control   ┆ 4    ┆ step_1 -> step_1  ┆ 845           ┆ 0             ┆ 845            │
│ Control   ┆ 5    ┆ step_3 -> step_1  ┆ 0             ┆ 769           ┆ 769            │
│ …         ┆ …    ┆ …                 ┆ …             ┆ …             ┆ …           

Started but never finished (incomplete)

In [17]:
incomplete = (
    visit_agg
    .filter(pl.col("completion_type") == "incomplete")
    .group_by("variation")
    .agg(pl.len().alias("n_incomplete_visits"))
    .sort("variation")
)

dropoff = (
    visit_agg
    .filter(pl.col("completion_type") == "incomplete")
    .with_columns(
        pl.col("step_ranks_list")
          .map_elements(
              lambda steps: max(int(s) for s in steps),
              return_dtype=pl.Int32
          )
          .alias("max_step_reached_rank")
    )
    .with_columns(
        pl.col("max_step_reached_rank")
          .map_elements(
              lambda r: RANK_TO_STEP.get(r, "unknown"),
              return_dtype=pl.String
          )
          .alias("furthest_step")
    )
    .group_by(["variation", "furthest_step"])
    .agg(pl.len().alias("n_visits"))
    .sort(["variation", "furthest_step"])
)

print(incomplete)
print()
print(dropoff)

shape: (2, 2)
┌───────────┬─────────────────────┐
│ variation ┆ n_incomplete_visits │
│ ---       ┆ ---                 │
│ str       ┆ u32                 │
╞═══════════╪═════════════════════╡
│ Control   ┆ 12373               │
│ Test      ┆ 14444               │
└───────────┴─────────────────────┘

shape: (2, 3)
┌───────────┬───────────────┬──────────┐
│ variation ┆ furthest_step ┆ n_visits │
│ ---       ┆ ---           ┆ ---      │
│ str       ┆ str           ┆ u32      │
╞═══════════╪═══════════════╪══════════╡
│ Control   ┆ unknown       ┆ 12373    │
│ Test      ┆ unknown       ┆ 14444    │
└───────────┴───────────────┴──────────┘


Understand how much time each step takes

In [18]:
df = df.with_columns(pl.col('date_time').str.to_datetime(format='%Y-%m-%d %H:%M:%S'))
df_sorted = df.sort(['visit_id', 'date_time'])

In [19]:
# Join completion_type into df_sorted
df_time = (
    df_sorted
    .join(
        visit_agg.select(["visit_id", "completion_type"]),
        on="visit_id",
        how="left"
    )
    .with_columns([
        pl.col("date_time").shift(-1).over("visit_id").alias("next_datetime"),
    ])
    .filter(pl.col("next_datetime").is_not_null())
    .with_columns(
        (pl.col("next_datetime") - pl.col("date_time"))
          .dt.total_seconds()
          .alias("seconds_in_step")
    )
)

# Function to summarise time per step
def time_per_step(df, completion):
    return (
        df
        .filter(pl.col("completion_type") == completion)
        .group_by(["Variation", "process_step"])
        .agg([
            pl.col("seconds_in_step").mean().alias("avg_seconds"),
            pl.col("seconds_in_step").median().alias("median_seconds"),
        ])
        .with_columns([
            (pl.col("avg_seconds") / 60).round(2).alias("avg_minutes"),
            (pl.col("median_seconds") / 60).round(2).alias("median_minutes"),
        ])
        .sort(["Variation", pl.col("process_step").replace(STEP_ORDER)])
    )

time_smooth = time_per_step(df_time, "smooth")
time_lumpy  = time_per_step(df_time, "lumpy")

print("── Smooth ──────────────────────────")
print(time_smooth)
print("\n── Lumpy ───────────────────────────")
print(time_lumpy)

── Smooth ──────────────────────────
shape: (8, 6)
┌───────────┬──────────────┬─────────────┬────────────────┬─────────────┬────────────────┐
│ Variation ┆ process_step ┆ avg_seconds ┆ median_seconds ┆ avg_minutes ┆ median_minutes │
│ ---       ┆ ---          ┆ ---         ┆ ---            ┆ ---         ┆ ---            │
│ str       ┆ str          ┆ f64         ┆ f64            ┆ f64         ┆ f64            │
╞═══════════╪══════════════╪═════════════╪════════════════╪═════════════╪════════════════╡
│ Control   ┆ start        ┆ 38.598124   ┆ 22.0           ┆ 0.64        ┆ 0.37           │
│ Control   ┆ step_1       ┆ 36.17678    ┆ 21.0           ┆ 0.6         ┆ 0.35           │
│ Control   ┆ step_2       ┆ 98.654682   ┆ 75.0           ┆ 1.64        ┆ 1.25           │
│ Control   ┆ step_3       ┆ 138.064232  ┆ 88.0           ┆ 2.3         ┆ 1.47           │
│ Test      ┆ start        ┆ 28.456488   ┆ 10.0           ┆ 0.47        ┆ 0.17           │
│ Test      ┆ step_1       ┆ 36.833753 

Completion rate = (smooth + lumpy visits) / total visits — anyone who reached confirm regardless of errors
Error rate = visits that had at least one error / total visits — based on the had_error flag already in visit_agg

In [20]:
# Completion Rate and Error Rate — Test vs Control
total_visits = visit_agg.group_by("variation").agg(pl.len().alias("total_visits"))

completion_rate = (
    visit_agg
    .filter(pl.col("completion_type").is_in(["smooth", "lumpy"]))
    .group_by("variation")
    .agg(pl.len().alias("completed_visits"))
    .join(total_visits, on="variation")
    .with_columns(
        (pl.col("completed_visits") / pl.col("total_visits") * 100)
          .round(2)
          .alias("completion_rate_%")
    )
    .sort("variation")
)

error_rate = (
    visit_agg
    .join(total_visits, on="variation")
    .group_by(["variation", "total_visits"])
    .agg(pl.col("had_error").cast(pl.Int32).sum().alias("visits_with_errors"))
    .with_columns(
        (pl.col("visits_with_errors") / pl.col("total_visits") * 100)
          .round(2)
          .alias("error_rate_%")
    )
    .sort("variation")
)

print("── Completion Rate ──────────────────────────")
print(completion_rate)
print("\n── Error Rate ───────────────────────────────")
print(error_rate)

── Completion Rate ──────────────────────────
shape: (2, 4)
┌───────────┬──────────────────┬──────────────┬───────────────────┐
│ variation ┆ completed_visits ┆ total_visits ┆ completion_rate_% │
│ ---       ┆ ---              ┆ ---          ┆ ---               │
│ str       ┆ u32              ┆ u32          ┆ f64               │
╞═══════════╪══════════════════╪══════════════╪═══════════════════╡
│ Control   ┆ 10896            ┆ 23269        ┆ 46.83             │
│ Test      ┆ 14285            ┆ 28729        ┆ 49.72             │
└───────────┴──────────────────┴──────────────┴───────────────────┘

── Error Rate ───────────────────────────────
shape: (2, 4)
┌───────────┬──────────────┬────────────────────┬──────────────┐
│ variation ┆ total_visits ┆ visits_with_errors ┆ error_rate_% │
│ ---       ┆ ---          ┆ ---                ┆ ---          │
│ str       ┆ u32          ┆ i32                ┆ f64          │
╞═══════════╪══════════════╪════════════════════╪══════════════╡
│ Control 

Trying to spot differences by age/gender. EDIT: GENDER COLUMN NEEDS TO BE CLEAN

Building demographics

In [21]:
client_demographics = (
    df_sorted
    .select(["client_id", "clnt_age", "gendr"])
    .unique(subset=["client_id"])  # one row per client
    .rename({"gendr": "gender", "clnt_age": "age"})
    .with_columns(
        pl.when(pl.col("age") <= 30).then(pl.lit("18-30"))
          .when(pl.col("age") <= 45).then(pl.lit("31-45"))
          .when(pl.col("age") <= 60).then(pl.lit("46-60"))
          .otherwise(pl.lit("60+"))
          .alias("age_group")
    )
)

visit_agg_demo = visit_agg.join(client_demographics, on="client_id", how="left")
print(visit_agg_demo.select(["client_id", "age", "gender", "age_group"]).head(5))

shape: (5, 4)
┌───────────┬──────┬────────┬───────────┐
│ client_id ┆ age  ┆ gender ┆ age_group │
│ ---       ┆ ---  ┆ ---    ┆ ---       │
│ i64       ┆ f64  ┆ str    ┆ str       │
╞═══════════╪══════╪════════╪═══════════╡
│ 3561384   ┆ 59.5 ┆ U      ┆ 46-60     │
│ 7338123   ┆ 23.5 ┆ M      ┆ 18-30     │
│ 105007    ┆ 35.0 ┆ F      ┆ 31-45     │
│ 5623007   ┆ 78.0 ┆ M      ┆ 60+       │
│ 4823947   ┆ 52.0 ┆ U      ┆ 46-60     │
└───────────┴──────┴────────┴───────────┘


Error rate by gender & age

In [22]:
error_rate_demo = (
    visit_agg_demo
    .group_by(["variation", "gender", "age_group"])
    .agg([
        pl.len().alias("total_visits"),
        pl.col("had_error").cast(pl.Int32).sum().alias("visits_with_errors"),
    ])
    .with_columns(
        (pl.col("visits_with_errors") / pl.col("total_visits") * 100)
          .round(2).alias("error_rate_%")
    )
    .sort(["variation", "gender", "age_group"])
)
print(error_rate_demo)

shape: (27, 6)
┌───────────┬────────┬───────────┬──────────────┬────────────────────┬──────────────┐
│ variation ┆ gender ┆ age_group ┆ total_visits ┆ visits_with_errors ┆ error_rate_% │
│ ---       ┆ ---    ┆ ---       ┆ ---          ┆ ---                ┆ ---          │
│ str       ┆ str    ┆ str       ┆ u32          ┆ i32                ┆ f64          │
╞═══════════╪════════╪═══════════╪══════════════╪════════════════════╪══════════════╡
│ Control   ┆ null   ┆ 60+       ┆ 3            ┆ 2                  ┆ 66.67        │
│ Control   ┆ F      ┆ 18-30     ┆ 671          ┆ 220                ┆ 32.79        │
│ Control   ┆ F      ┆ 31-45     ┆ 1670         ┆ 497                ┆ 29.76        │
│ Control   ┆ F      ┆ 46-60     ┆ 2570         ┆ 872                ┆ 33.93        │
│ Control   ┆ F      ┆ 60+       ┆ 2456         ┆ 849                ┆ 34.57        │
│ …         ┆ …      ┆ …         ┆ …            ┆ …                  ┆ …            │
│ Test      ┆ U      ┆ 18-30     ┆ 2795

Completion rate by gender & age

In [23]:
completion_rate_demo = (
    visit_agg_demo
    .with_columns(
        pl.col("completion_type").is_in(["smooth", "lumpy"]).cast(pl.Int32)
          .alias("is_completed")
    )
    .group_by(["variation", "gender", "age_group"])
    .agg([
        pl.len().alias("total_visits"),
        pl.col("is_completed").sum().alias("completed_visits"),
    ])
    .with_columns(
        (pl.col("completed_visits") / pl.col("total_visits") * 100)
          .round(2).alias("completion_rate_%")
    )
    .sort(["variation", "gender", "age_group"])
)
print(completion_rate_demo)

shape: (27, 6)
┌───────────┬────────┬───────────┬──────────────┬──────────────────┬───────────────────┐
│ variation ┆ gender ┆ age_group ┆ total_visits ┆ completed_visits ┆ completion_rate_% │
│ ---       ┆ ---    ┆ ---       ┆ ---          ┆ ---              ┆ ---               │
│ str       ┆ str    ┆ str       ┆ u32          ┆ i32              ┆ f64               │
╞═══════════╪════════╪═══════════╪══════════════╪══════════════════╪═══════════════════╡
│ Control   ┆ null   ┆ 60+       ┆ 3            ┆ 3                ┆ 100.0             │
│ Control   ┆ F      ┆ 18-30     ┆ 671          ┆ 337              ┆ 50.22             │
│ Control   ┆ F      ┆ 31-45     ┆ 1670         ┆ 854              ┆ 51.14             │
│ Control   ┆ F      ┆ 46-60     ┆ 2570         ┆ 1188             ┆ 46.23             │
│ Control   ┆ F      ┆ 60+       ┆ 2456         ┆ 951              ┆ 38.72             │
│ …         ┆ …      ┆ …         ┆ …            ┆ …                ┆ …                 │
│ Test

Smooth vs Lumpy by gender & age

In [24]:
smooth_lumpy_demo = (
    visit_agg_demo
    .filter(pl.col("completion_type").is_in(["smooth", "lumpy"]))
    .group_by(["variation", "gender", "age_group", "completion_type"])
    .agg(pl.len().alias("n_visits"))
    .sort(["variation", "gender", "age_group", "completion_type"])
)
print(smooth_lumpy_demo)

shape: (52, 5)
┌───────────┬────────┬───────────┬─────────────────┬──────────┐
│ variation ┆ gender ┆ age_group ┆ completion_type ┆ n_visits │
│ ---       ┆ ---    ┆ ---       ┆ ---             ┆ ---      │
│ str       ┆ str    ┆ str       ┆ str             ┆ u32      │
╞═══════════╪════════╪═══════════╪═════════════════╪══════════╡
│ Control   ┆ null   ┆ 60+       ┆ lumpy           ┆ 2        │
│ Control   ┆ null   ┆ 60+       ┆ smooth          ┆ 1        │
│ Control   ┆ F      ┆ 18-30     ┆ lumpy           ┆ 114      │
│ Control   ┆ F      ┆ 18-30     ┆ smooth          ┆ 223      │
│ Control   ┆ F      ┆ 31-45     ┆ lumpy           ┆ 267      │
│ …         ┆ …      ┆ …         ┆ …               ┆ …        │
│ Test      ┆ U      ┆ 31-45     ┆ smooth          ┆ 864      │
│ Test      ┆ U      ┆ 46-60     ┆ lumpy           ┆ 559      │
│ Test      ┆ U      ┆ 46-60     ┆ smooth          ┆ 664      │
│ Test      ┆ U      ┆ 60+       ┆ lumpy           ┆ 345      │
│ Test      ┆ U      ┆ 60

In [25]:
print(
    client_demographics
    .join(
        df_sorted.select(["client_id", "Variation"]).unique(),
        on="client_id",
        how="left"
    )
    .group_by(["Variation", "gender", "age_group"])
    .agg(pl.len().alias("n_clients"))
    .sort(["Variation", "n_clients"], descending=[False, True])
)



shape: (27, 4)
┌───────────┬────────┬───────────┬───────────┐
│ Variation ┆ gender ┆ age_group ┆ n_clients │
│ ---       ┆ ---    ┆ ---       ┆ ---       │
│ str       ┆ str    ┆ str       ┆ u32       │
╞═══════════╪════════╪═══════════╪═══════════╡
│ Control   ┆ M      ┆ 46-60     ┆ 1982      │
│ Control   ┆ F      ┆ 46-60     ┆ 1977      │
│ Control   ┆ U      ┆ 18-30     ┆ 1848      │
│ Control   ┆ M      ┆ 60+       ┆ 1782      │
│ Control   ┆ F      ┆ 60+       ┆ 1749      │
│ …         ┆ …      ┆ …         ┆ …         │
│ Test      ┆ U      ┆ 60+       ┆ 1310      │
│ Test      ┆ M      ┆ 18-30     ┆ 881       │
│ Test      ┆ F      ┆ 18-30     ┆ 685       │
│ Test      ┆ null   ┆ 60+       ┆ 6         │
│ Test      ┆ X      ┆ 18-30     ┆ 1         │
└───────────┴────────┴───────────┴───────────┘
